In [13]:
!pip install plotly
!pip install plotly[express]
!pip install --upgrade nbformat

zsh:1: no matches found: plotly[express]


25/10/08 23:05:20 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:700)
	at org.apache.spark.storage.BlockManagerMasterE

In [14]:
!pyspark --version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/08 23:05:22 WARN Utils: Your hostname, Drashis-MacBook-Air-8.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.229 instead (on interface en0)
25/10/08 23:05:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.1
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 21.0.8
Branch HEAD
Compiled by user runner on 2025-09-02T03:10:51Z
Revision 29434ea766b0fc3c3bf6eaadb43a8f931133649e
Url https://github.com/apache/spark
Type --help for more information.


In [1]:
from pyspark.sql import SparkSession
import plotly.express as px

In [2]:
spark = SparkSession.builder \
    .appName("FlightDelay") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/08 23:05:52 WARN Utils: Your hostname, Drashis-MacBook-Air-8.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.229 instead (on interface en0)
25/10/08 23:05:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/08 23:05:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 51876)
Traceback (most recent call last):
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/site-packages/pyspark/accumu

In [3]:
df = spark.read.csv( "/Users/drashi/Documents/UMBC-DATA606-Capstone/Data/combined_flight_data.csv", header=True, inferSchema=True)

### PHASE 1 — DATA CLEANING & SCHEMA SETUP

#### Basic Setup & Schema Inspection

In [15]:
from pyspark.sql import functions as F

# Show record count and schema
print(f"Total records: {df.count():,}")
df.printSchema()

Total records: 17,373,636
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: double (nullable = true)
 |-- OriginAirportID: integer (nullable = true)
 |-- OriginAirportSeqID: integer (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateFips: integer (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginWac: integer (nullable = true)
 |-- DestAirportID: integer (nullable = true

In [16]:
# Quick look at data
df.show(5, truncate=False)

+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+--------------+----------------+----

#### Filter Out Cancelled & Diverted Flights

Cancelled or diverted flights can distort delay calculations. We will have only valid, completed flights.

In [17]:
df = df.filter((F.col("Cancelled") == 0) & (F.col("Diverted") == 0))

#### Drop Irrelevant or Redundant Columns

In [18]:
#### Drop Irrelevant or Redundant Columns

cols_to_drop = [
    'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestStateFips', 'DestStateName', 'DestWac',
    'Cancelled', 'CancellationCode', 'Diverted',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay', 'DivDistance'
]

df = df.drop(*cols_to_drop)

#### Count Null (and Missing) Values per Column

In [22]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# numeric and string columns
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

# For string columns: check both NULL and empty string
# For numeric columns: check only NULL
null_counts_expr = [
    F.count(
        F.when(
            F.col(c).isNull() | ((F.col(c) == '') if c in string_cols else F.lit(False)),
            c
        )
    ).alias(c)
    for c in df.columns
]

null_counts = df.select(null_counts_expr)
null_counts.show(truncate=False)


+----+-------+-----+----------+---------+----------+-----------------+-------------------------------+------+--------------+-----------+----+------------+---------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+--------------+-----------------+-------+-------+--------+-------------+-----------+-------------+----------------+------------+--------------+----------------+-------------+-----------+-----------+-------------+----------------+------------+--------------+----------------+-------------+-----------+-----------+-------------+----------------+------------+--------------+----------------+-------------+-----------+-----------+-------------+----------------+------------+--------------+----------------+-------------+-----------+-----------+-------------+----------------+------------+--------------+----------------+-------------+----

Dropping the Div* Columns as these columns (Div1Airport, Div2Airport, Div3AirportID, etc.) store information only for flights that were diverted and they didn’t land at their scheduled destination.

In [23]:
# Drop all Div columns (Div1Airport, Div2Airport, etc.)
div_cols = [c for c in df.columns if c.startswith("Div")]
df = df.drop(*div_cols)

#### Handle Missing Values 

Our goals here are to:
1. Critical fields (like Reporting_Airline, Origin, Dest, FlightDate): drop rows if null.
2. Numeric time/delay fields (DepDelay, ArrDelay, TaxiOut, TaxiIn, etc.): fill with 0 (because 0 = “no delay” or “not applicable”).
3. Extra or empty string columns: drop or clean. -

In [24]:
# 1️. Drop rows missing critical info
critical_cols = ['Reporting_Airline', 'Origin', 'Dest', 'FlightDate']
df = df.dropna(subset=critical_cols)

In [25]:
# 2️. Replace nulls in numeric columns with 0
numeric_cols = [
    'DepDelay', 'DepDelayMinutes', 'ArrDelay', 'ArrDelayMinutes',
    'TaxiOut', 'TaxiIn', 'CRSElapsedTime', 'ActualElapsedTime',
    'AirTime', 'Distance', 'DistanceGroup', 'Flights'
]
df = df.fillna(0, subset=[c for c in numeric_cols if c in df.columns])

In [26]:
# 3️. Replace nulls in binary/delay indicators with 0
binary_cols = ['DepDel15', 'ArrDel15']
df = df.fillna(0, subset=[c for c in binary_cols if c in df.columns])

In [27]:
# 4️. Drop any unnamed or completely empty columns
df = df.drop('Unnamed: 109') if 'Unnamed: 109' in df.columns else df

In [28]:
# recheck for remaining nulls
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show(truncate=False)

+----+-------+-----+----------+---------+----------+-----------------+-------------------------------+------+--------------+-----------+----+------------+---------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+--------------+-----------------+-------+-------+--------+-------------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|Flight_Number_Reporting_Airline|Origin|OriginCityName|OriginState|Dest|DestCityName|DestState|CRSDepTime|DepTime|DepDelay|DepDelayMinutes|DepDel15|DepartureDelayGroups|DepTimeBlk|TaxiOut|WheelsOff|WheelsOn|TaxiIn|CRSArrTime|ArrTime|ArrDelay|ArrDelayMinutes|ArrDel15|ArrivalDelayGroups|ArrTimeBlk|CRSElapsedTime|ActualElapsedTime|AirTime|Flights|Distance|DistanceGroup|
+----+-------+-----+----------+---------+----------+-----------------+-------------------------------+------+-------------

In [30]:
df.select('DepDelay', 'ArrDelay', 'TaxiOut', 'TaxiIn').summary().show()
print(f" Final record count after cleaning: {df.count():,}")

+-------+------------------+------------------+------------------+-----------------+
|summary|          DepDelay|          ArrDelay|           TaxiOut|           TaxiIn|
+-------+------------------+------------------+------------------+-----------------+
|  count|          17094473|          17094473|          17094473|         17094473|
|   mean|12.492518254291898|7.0706265118556155|17.832524992142197|8.254146179294326|
| stddev| 55.62474591227864|  57.8240421974776| 9.715341468392428|6.718026205211478|
|    min|             -99.0|            -128.0|               1.0|              0.0|
|    25%|              -6.0|             -15.0|              12.0|              4.0|
|    50%|              -2.0|              -6.0|              15.0|              6.0|
|    75%|               9.0|              10.0|              21.0|             10.0|
|    max|            4413.0|            4405.0|             291.0|            444.0|
+-------+------------------+------------------+------------------

 Final record count after cleaning: 17,094,473


### PHASE 3 — EXPLORATORY DATA ANALYSIS

#### Distribution of Departure and Arrival Delays

Goal: Understand how departure and arrival delays are distributed — identify whether most flights are on time, early, or significantly delayed.

Instead of loading every record, we’ll:
- Group delays into bins (5-minute intervals)
- Count flights per bin (using PySpark)
- Convert only the small summary (a few hundred rows) to Pandas
- Visualize with Plotly

In [ ]:
from pyspark.sql import functions as F
import plotly.express as px

#  delay bins (every 5 minutes)
# Floor division groups delays like 0-4, 5-9, 10-14, etc.
dep_delay_hist = (
    df.withColumn("DepDelayBin", (F.floor(F.col("DepDelay") / 5) * 5))
      .groupBy("DepDelayBin")
      .agg(F.count("*").alias("FlightCount"))
      .orderBy("DepDelayBin")
)

arr_delay_hist = (
    df.withColumn("ArrDelayBin", (F.floor(F.col("ArrDelay") / 5) * 5))
      .groupBy("ArrDelayBin")
      .agg(F.count("*").alias("FlightCount"))
      .orderBy("ArrDelayBin")
)

# Convert only aggregated bins to Pandas (few hundred rows — very light)
dep_pdf = dep_delay_hist.toPandas()
arr_pdf = arr_delay_hist.toPandas()

# Plot aggregated distributions
fig_dep = px.bar(
    dep_pdf, 
    x="DepDelayBin", 
    y="FlightCount",
    title="Distribution of Departure Delays (All Data, 5-Minute Bins)",
    labels={"DepDelayBin": "Departure Delay (minutes)", "FlightCount": "Number of Flights"},
)
fig_dep.update_layout(bargap=0.1)
fig_dep.show()

fig_arr = px.bar(
    arr_pdf, 
    x="ArrDelayBin", 
    y="FlightCount",
    title="Distribution of Arrival Delays (All Data, 5-Minute Bins)",
    labels={"ArrDelayBin": "Arrival Delay (minutes)", "FlightCount": "Number of Flights"},
)
fig_arr.update_layout(bargap=0.1)
fig_arr.show()

Interpretation:

Departure Delay Distribution
- Most flights depart on time or within ±10 minutes.
- The distribution is right-skewed, showing a few flights with very large delays (100+ minutes).
- Indicates that the airline system is generally reliable, but rare events (weather, congestion) cause long delays.

 Arrival Delay Distribution
- Follows a similar pattern but slightly more spread out.
- Suggests that departure delays often carry over to arrival delays, with added effects from air traffic or weather.
- A few extreme outliers (multi-hour delays) heavily impact averages.

#### Average Delay by Month

Goal: Identify which months experience the highest and lowest average departure and arrival delays to uncover seasonal trends in flight performance.

In [ ]:
from pyspark.sql import functions as F
import plotly.express as px

# Compute monthly average delays for both departure and arrival
avg_delay_by_month = (
    df.groupBy("Month")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy("Month")
)

# Convert aggregated data (12 rows) to Pandas for plotting
avg_delay_month_pdf = avg_delay_by_month.toPandas()

# Plot line chart for monthly delay trends
fig = px.line(
    avg_delay_month_pdf,
    x="Month",
    y=["AvgDepDelay", "AvgArrDelay"],
    markers=True,
    title="Average Departure and Arrival Delay by Month",
    labels={"value": "Average Delay (minutes)", "Month": "Month", "variable": "Delay Type"}
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

Interpretation:
- Peak delays occur in June and July, likely due to summer travel congestion and thunderstorms.
- January also shows elevated delays, reflecting winter weather disruptions.
- Lowest delays appear around October–November, indicating smoother seasonal operations.
- Arrival delays consistently track slightly below departure delays, suggesting that flights often make up some lost time en route.

Overall Insight:
- Flight delays show clear seasonal patterns: highest in summer and winter, lowest in fall,  highlighting the strong influence of weather and travel demand on delay behavior.

#### Average Delay by Day of Week

Goal: Identify which days of the week experience higher or lower flight delays — useful for detecting operational or demand-based trends (e.g., business travel vs leisure peaks).

In [34]:
from pyspark.sql import functions as F
import plotly.express as px

# Compute average delays per day of week
avg_delay_by_dow = (
    df.groupBy("DayOfWeek")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy("DayOfWeek")
)

# Convert to Pandas for Plotly visualization (only 7 rows)
avg_delay_dow_pdf = avg_delay_by_dow.toPandas()

# Create bar chart
fig = px.bar(
    avg_delay_dow_pdf,
    x="DayOfWeek",
    y=["AvgDepDelay", "AvgArrDelay"],
    barmode="group",
    title="Average Departure and Arrival Delay by Day of Week",
    labels={"value": "Average Delay (minutes)", "DayOfWeek": "Day of Week", "variable": "Delay Type"}
)

# Improve readability
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[1, 2, 3, 4, 5, 6, 7],
        ticktext=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    )
)
fig.show()

Interpretation
- Highest delays: Friday and Sunday → heavy air traffic from business and weekend travel.
- Lowest delays: Tuesday and Wednesday → lighter schedules, fewer congestion issues.
- Arrival delays are consistently lower than departure delays → flights often make up time in the air.

Insight:
Delays clearly vary by weekday, showing that traffic volume and scheduling patterns strongly influence punctuality.

#### Average Delay by Hour of Day
Goal: Understand how flight delays vary throughout the day — from early morning departures to late-night flights — to detect daily congestion or cascading delay effects.

In [35]:
from pyspark.sql import functions as F
import plotly.express as px

# Extract scheduled departure hour (0–23) from CRSDepTime
df = df.withColumn("DepHour", (F.col("CRSDepTime") / 100).cast("int"))

# Compute average departure and arrival delays per hour
avg_delay_by_hour = (
    df.groupBy("DepHour")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy("DepHour")
)

# Convert to Pandas for plotting
avg_delay_hour_pdf = avg_delay_by_hour.toPandas()

# Plot the trends
fig = px.line(
    avg_delay_hour_pdf,
    x="DepHour",
    y=["AvgDepDelay", "AvgArrDelay"],
    markers=True,
    title="Average Departure and Arrival Delay by Hour of Day",
    labels={
        "DepHour": "Scheduled Departure Hour (Local Time)",
        "value": "Average Delay (minutes)",
        "variable": "Delay Type"
    }
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

Interpretation

1. Early Morning (5–8 AM):
    - Flights show the lowest average delays — operations start fresh with minimal congestion.
2. Midday to Evening (12–20 hrs):
    - Delays gradually increase, peaking in the late afternoon and early evening as the air traffic network becomes more congested and earlier delays cascade through the schedule.
3. Late Night (after 22 hrs):
    - Average delays drop sharply, indicating lighter traffic and recovery time before the next day’s schedule.
4. Departure vs Arrival:
    - Arrival delays mirror departure delays but remain slightly lower, showing that flights often recover some time midair.

Overall Insight
- Delays accumulate as the day progresses — a clear sign of the “ripple effect” in flight scheduling. Morning flights are the most reliable, while evening departures face the highest risk of delay.

#### Average Delay by Airline (with names)
Goal: Compare average departure and arrival delays across different airlines to assess carrier-level performance and reliability.

In [37]:
df.select("Reporting_Airline").distinct().orderBy("Reporting_Airline").show(truncate=False)

+-----------------+
|Reporting_Airline|
+-----------------+
|9E               |
|AA               |
|AS               |
|B6               |
|DL               |
|F9               |
|G4               |
|HA               |
|MQ               |
|NK               |
|OH               |
|OO               |
|UA               |
|WN               |
|YX               |
+-----------------+



In [39]:
from pyspark.sql import functions as F
import plotly.express as px

# Aggregate average delays by airline
avg_delay_by_airline = (
    df.groupBy("Reporting_Airline")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy("Reporting_Airline")
)

# Convert to Pandas
avg_delay_airline_pdf = avg_delay_by_airline.toPandas()

# Airline code-to-name mapping
airline_name_map = {
    "9E": "Endeavor Air",
    "AA": "American Airlines",
    "AS": "Alaska Airlines",
    "B6": "JetBlue Airways",
    "DL": "Delta Air Lines",
    "F9": "Frontier Airlines",
    "G4": "Allegiant Air",
    "HA": "Hawaiian Airlines",
    "MQ": "Envoy Air",
    "NK": "Spirit Airlines",
    "OH": "PSA Airlines",
    "OO": "SkyWest Airlines",
    "UA": "United Airlines",
    "WN": "Southwest Airlines",
    "YX": "Republic Airways"
}

# Add airline name column
avg_delay_airline_pdf["AirlineName"] = avg_delay_airline_pdf["Reporting_Airline"].map(airline_name_map)

# Create interactive bar chart
fig = px.bar(
    avg_delay_airline_pdf,
    x="Reporting_Airline",  # Show airline codes on x-axis
    y=["AvgDepDelay", "AvgArrDelay"],
    barmode="group",
    title="Average Departure and Arrival Delay by Airline",
    labels={
        "Reporting_Airline": "Airline Code",
        "value": "Average Delay (minutes)",
        "variable": "Delay Type"
    },
    hover_data={  
        "AirlineName": True,
        "FlightCount": True,
        "Reporting_Airline": False  # Don’t duplicate in hover
    }
)

# Layout customization
fig.update_layout(
    xaxis_tickangle=-45,
    bargap=0.2,
    legend_title_text="Delay Type",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

Interpretation
- Frontier (F9) and American (AA) show the highest average delays, indicating consistent operational challenges.
- Delta (DL), Alaska (AS), and Hawaiian (HA) perform best, with the lowest delays.
- Low-cost carriers (like Frontier and Spirit) tend to face longer delays, likely due to tighter schedules.
- Arrival delays are generally slightly lower than departure delays, showing that airlines often recover time mid-flight.

Insights
- There’s a clear variation in performance by carrier — major legacy airlines (Delta, Alaska, Hawaiian) are generally more punctual, while some budget and regional airlines experience longer average delays, highlighting operational trade-offs between efficiency and cost.

#### Average Delay by Origin Airport (Top 15 Airports)
Goal: Identify which airports experience the highest average delays, for both departures and arrivals, to uncover congestion or regional effects.

In [43]:
from pyspark.sql import functions as F
import plotly.express as px

# Compute average delays per origin airport
avg_delay_by_origin = (
    df.groupBy("Origin", "OriginCityName")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy(F.desc("AvgDepDelay"))
      .limit(15)
)

# Convert to Pandas
avg_delay_origin_pdf = avg_delay_by_origin.toPandas()

# Create grouped bar chart
fig = px.bar(
    avg_delay_origin_pdf,
    x="Origin",  # Airport code on x-axis
    y=["AvgDepDelay", "AvgArrDelay"],
    barmode="group",
    title="Top 15 Airports by Average Departure and Arrival Delay",
    labels={
        "Origin": "Airport Code",
        "value": "Average Delay (minutes)",
        "variable": "Delay Type"
    },
    hover_data={
        "OriginCityName": True,  # show city and state on hover
        "FlightCount": True
    }
)

# Layout enhancements
fig.update_layout(
    xaxis_tickangle=-45,
    bargap=0.2,
    legend_title_text="Delay Type",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

Interpretations:
- SMX (Santa Maria, CA) and MGW (Morgantown, WV) experience the longest delays, while others like HTS and HOB show moderate ones.
- Departure delays are slightly higher than arrival delays, indicating that flights often recover time mid-air.
- Smaller regional airports tend to have higher delays, likely due to limited capacity and operational constraints.

Insight: 
- Major hubs usually manage delays better, while smaller airports face more disruptions — valuable for identifying delay-prone routes and optimizing scheduling.

#### Average Delay by Destination Airport (Top 15)

Goal: To analyze which destination airports experience the highest average departure and arrival delays

In [44]:
from pyspark.sql import functions as F
import plotly.express as px

# Compute average delays per destination airport
avg_delay_by_dest = (
    df.groupBy("Dest", "DestCityName")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy(F.desc("AvgArrDelay"))
      .limit(15)
)

# Convert to Pandas for plotting
avg_delay_dest_pdf = avg_delay_by_dest.toPandas()

# Create grouped bar chart
fig = px.bar(
    avg_delay_dest_pdf,
    x="Dest",
    y=["AvgDepDelay", "AvgArrDelay"],
    barmode="group",
    title="Top 15 Destination Airports by Average Departure and Arrival Delay",
    labels={
        "Dest": "Airport Code",
        "value": "Average Delay (minutes)",
        "variable": "Delay Type"
    },
    hover_data={
        "DestCityName": True,
        "FlightCount": True
    }
)

# Layout customization
fig.update_layout(
    xaxis_tickangle=-45,
    bargap=0.2,
    legend_title_text="Delay Type",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

Interpretation:
- HOB (Hobbs, NM) and PVU (Provo, UT) have the highest average delays, indicating frequent arrival disruptions.
- Departure delays are generally a bit higher than arrival delays, meaning flights recover some lost time mid-air.
- Smaller regional destinations (e.g., MGW, FMN) tend to face more severe average delays than major hubs, likely due to limited air traffic capacity and weather sensitivity.

Insight:
- Flights heading to smaller or less-connected destinations are more delay-prone, suggesting that route structure and airport resources significantly impact overall on-time performance.

#### Correlation Analysis

Goal: To examine relationships between key numerical features — such as distance, airtime, departure delay, and arrival delay — in order to identify which variables are most influential for predicting delays.


Why Sampling Works for Correlation Analysis?
- Correlation values (like Pearson’s r) stabilize very quickly as sample size grows — because they depend on relative variation, not exact counts.
That means you don’t need the full dataset to get highly accurate correlations.

In [54]:
from pyspark.sql import functions as F
import pandas as pd
import plotly.express as px

# Enable Arrow optimization for faster conversion
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# Select numeric columns
numeric_cols = [
    "DepDelay", "ArrDelay", "AirTime", "Distance",
    "TaxiOut", "TaxiIn", "CRSElapsedTime", "ActualElapsedTime"
]

# Take a 5% random sample for correlation analysis
sample_df = (
    df.select(numeric_cols)
      .sample(fraction=0.05, seed=42)
      .na.drop()
)

print("Converting 5% Spark sample to Pandas using Arrow...")
numeric_pdf = sample_df.toPandas()
print(f" Conversion done — shape: {numeric_pdf.shape}")

# Compute correlation matrix
corr_df = numeric_pdf.corr()

# Plot heatmap
fig = px.imshow(
    corr_df,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Correlation Heatmap ",
    width=950,        
    height=800  
)
fig.update_layout(
    xaxis_title="Feature",
    yaxis_title="Feature",
    coloraxis_colorbar=dict(title="Correlation Coefficient"),
)
fig.show()

Converting 5% Spark sample to Pandas using Arrow...


 Conversion done — shape: (854600, 8)


25/10/09 04:53:41 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 994351 ms exceeds timeout 120000 ms
25/10/09 04:53:41 WARN SparkContext: Killing executors is not supported by current scheduler.
25/10/09 04:53:49 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

Interpretation:
1. Departure Delay (DepDelay) and Arrival Delay (ArrDelay) are strongly correlated (r ≈ 0.97).
    - Flights that depart late almost always arrive late — a direct propagation effect.
2. AirTime, Distance, CRSElapsedTime, and ActualElapsedTime are very highly correlated (r ≈ 0.97–0.99).
    - These all measure aspects of flight duration and are essentially interdependent, so only one or two should be kept to avoid multicollinearity.
3. TaxiOut shows mild correlation with delays (ArrDelay: 0.19).
    - Longer taxi-out times may slightly contribute to arrival delays.
4. TaxiIn has a very weak correlation (≈ 0.1 or less) with both departure and arrival delays.
    - Taxi-in duration is mostly influenced by airport congestion after landing rather than schedule adherence.

Insights
- Strong multicollinearity exists among time-related variables (AirTime, Distance, ElapsedTime).
These should be handled carefully during modeling (e.g., through feature selection or PCA).
- DepDelay is the most influential predictor for ArrDelay — confirming that departure performance drives arrival outcomes.
- Operational ground factors (TaxiOut/TaxiIn) contribute minimally to overall delay variance.

#### EDA Summary & Feature Insights

**Overall Findings**
- Most flights have small to moderate delays; extreme delays are rare but impactful.
- Delays peak during summer months (June–July) and are lowest in winter (Nov–Jan).
- Fridays and weekends see higher average delays, likely due to heavier traffic
- Late evening flights (17:00–22:00) experience higher delays, indicating schedule ripple effects.
- Some airlines (e.g., Frontier (F9), American (AA)) show higher average delays, while others (Alaska (AS), SkyWest (OO)) perform better.
- few origin airports contribute disproportionately to delays, reflecting congestion or operational inefficiency.

**Correlation Insights**
- DepDelay ↔ ArrDelay: Very strong correlation (r ≈ 0.97) — late departures lead to late arrivals.
- AirTime, Distance, CRSElapsedTime, and ActualElapsedTime are highly correlated (r ≈ 0.97–0.99) → only one or two should be retained.
- TaxiOut has a weak-to-moderate positive correlation with ArrDelay (r ≈ 0.19), suggesting minor ground delay impact.
- TaxiIn shows almost no relation to delays — can be deprioritized.

Key Predictors for Modeling:

| Category             | Selected Features                                 |
| -------------------- | ------------------------------------------------- |
| **Temporal**         | Month, DayOfWeek, Quarter, CRSDepTime (or Hour)   |
| **Operational**      | Distance, AirTime (or ActualElapsedTime), TaxiOut |
| **Location**         | Origin, Dest                                      |
| **Airline**          | Reporting_Airline                                 |
| **Historical Delay** | DepDelay (for predicting ArrDelay)                |

To Drop or Merge:
- Highly collinear: CRSElapsedTime, ActualElapsedTime, Distance → keep one representative.
- Weak correlation: TaxiIn.
- Uninformative: Div1–Div5 fields, Unnamed: 109, redundant IDs.


Target Variables:

| Stage                 | Goal                                               | Target Variables       | Type         |
| --------------------- | -------------------------------------------------- | ---------------------- | ------------ |
| **1. Classification** | Predict if a flight is delayed (≥15 min)           | `DepDel15`, `ArrDel15` | Binary (0/1) |
| **2. Regression**     | Predict the delay duration (in minutes) if delayed | `DepDelay`, `ArrDelay` | Continuous   |


Summary:

- EDA confirms that time factors, airline, airport, and departure performance are the strongest predictors of flight delays.
- The modeling phase will use these insights to build a two-stage predictive framework — first detecting delay occurrence, then estimating delay duration for delayed flights.